In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Generate flow that highly depends on yesterday's flow (phi=0.8)
# river do not work independent


import scipy.signal

np.random.seed(42)
T = 100
rainfall = np.random.uniform(0, 20, T)
runoff_coeff = 0.5

# prepare today's input (rainfall effect + random noise).
noise = np.random.normal(0, 1, T)
exog = runoff_coeff * rainfall + noise

exog[0] = 0  # Anchor first day so we match the base flow cleanly i.e. the first flow will only be baseline10
#  we add the baseline 10 later

# lfilter pushes the recursion to C: y[t] = 0.8*y[t-1] + exog[t]

# The 'a' array is the left side constants of the equation: 1 times y[t] - 0.8 times y[t-1] = exog[t]
flow_centered = scipy.signal.lfilter(b=[1.0], a=[1.0, -0.8], x=exog)
flow_true = flow_centered + 10  # Shift back up to resting baseline (10)

# up till now we have just made the synthetic data

# a simple standard bayesian regression model with no memory
with pm.Model() as model_static:

    b_rain = pm.HalfNormal("b_rain", sigma=2)
    intercept = pm.Normal("intercept", mu=10, sigma=5)
    sigma = pm.HalfNormal("sigma", sigma=5)
    mu = pm.Deterministic("mu", intercept + b_rain * rainfall)
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=flow_true)
    trace_static = pm.sample(1000, tune=1000, cores=1, random_seed=42)

    prior_checks = pm.sample_prior_predictive(draws=1000, random_seed=42)

    trace_static = pm.sample(draws=2000, tune=1000, chains=2, random_seed=42, progressbar=False)

static_preds = trace_static.posterior["mu"].mean(dim=["chain", "draw"])
residuals = flow_true - static_preds

plt.acorr(residuals, maxlags=20)
plt.title("M2 Static Model Residuals: MASSIVE AUTOCORRELATION")
plt.show()

In [ ]:
plt.plot(flow_true, label="Observed Physical Reality", color="black", linewidth=2)
plt.plot(static_preds, label="Static M2 Inference (No Memory)", color="red", alpha=0.8)
plt.legend()
plt.title("The Amateur's Mistake: Missing the Physical Decay")
plt.show()

In [ ]:
# The Invariant: Water takes time to leave the system. Yesterday's flow dictates today's baseline.
# Financial Liability: Failing to model autoregression (AR) in hydrology means underestimating
# the duration of flood events, leading to catastrophic under-design of diversion tunnels.

# Mean-center continuous data to eliminate the intercept trap.
# This decouples the baseline magnitude from the variance and AR coefficients.
flow_centered = flow_true - flow_true.mean()
rain_centered = rainfall - rainfall.mean()

with pm.Model() as m3_arx:
    #  We expect positive autocorrelation (water doesn't instantly vanish)
    phi = pm.Normal("phi", mu=0.5, sigma=0.5)

    # Rainfall translates directly to flow, but we leave the prior broad to let data speak
    b_rain = pm.Normal("b_rain", mu=0, sigma=2)

    # System noise: What variations in flow are not explained by rain or yesterday's flow?
    sigma = pm.HalfNormal("sigma", sigma=5)

    # The Exogenous forcing factor for the sequence
    mu_exo = b_rain * rain_centered

    # PyMC variables cannot be passed to 'observed' in pm.AR.
    # ARX relationship:
    # E[y_t] = phi * y_{t-1} + beta * X_t

    mu = pm.Deterministic("mu", phi * flow_centered[:-1] + mu_exo[1:])


    y = pm.Normal("y",
                  mu=mu,
                  sigma=sigma,
                  observed=flow_centered[1:])

    trace_arx = pm.sample(draws=1000,
                          tune=1000,
                          cores=1,
                          target_accept=0.95,
                          random_seed=42,
                          progressbar=False)

    pm.compute_log_likelihood(trace_arx)


print(az.summary(trace_arx, var_names=["phi", "b_rain"]))

In [ ]:
#erratic river with no persistence
flow_true_wn = np.zeros(T)
flow_true_wn[0] = 10 + 0.5 * rainfall[0] + np.random.normal(0, 1)

for t in range(1, T):
    # phi is 0.0 (flow[t-1] has zero impact on flow[t])
    flow_true_wn[t] = 10 + 0.0 * (flow_true_wn[t-1] - 10) + 0.5 * rainfall[t] + np.random.normal(0, 1)

# Mean-center continuous data
flow_centered_wn = flow_true_wn - flow_true_wn.mean()

with pm.Model() as m3_arx_wn:

    phi_wn = pm.Normal("phi", mu=0.5, sigma=0.5)


    b_rain_wn = pm.Normal("b_rain", mu=0, sigma=2)


    sigma_wn = pm.HalfNormal("sigma", sigma=5)


    mu_exo_wn = b_rain_wn * rain_centered
    mu_wn = pm.Deterministic("mu", phi_wn * flow_centered_wn[:-1] + mu_exo_wn[1:])

    y_wn = pm.Normal("y", mu=mu_wn, sigma=sigma_wn, observed=flow_centered_wn[1:])

    # Sample
    trace_arx_wn = pm.sample(
        1000,
        tune=1000,
        cores=1,
        target_accept=0.95,
        random_seed=42,
        progressbar=False
    )


print(az.summary(trace_arx_wn, var_names=["phi", "b_rain"]))

In [ ]:
az.plot_posterior(trace_arx_wn, var_names=["phi", "b_rain"], hdi_prob=0.95)
plt.title("Forensic Validation: HDI Confirming Zero Autoregression (White Noise)")
plt.show()

In [ ]:
with m3_arx_wn:
    pm.sample_posterior_predictive(trace_arx_wn, extend_inferencedata=True, random_seed=42)
az.plot_ppc(trace_arx_wn, num_pp_samples=100, figsize=(10, 5), colors=["#1f77b4", "#ff7f0e", "#2ca02c"], kind="cumulative")

In [ ]:
ar1_preds = trace_arx.posterior["mu"].mean(dim=["chain", "draw"])
plt.acorr(flow_centered[1:] - ar1_preds, maxlags=20)
plt.title("Forensic Audit: AR(1) Model Residuals (White Noise Achieved)")

In [ ]:
plt.plot(flow_true[1:], label="Observed Physical Reality", color="black", linewidth=2)
plt.plot(ar1_preds + flow_true.mean(), label="Bayesian AR(1) Inference", color="red", alpha=0.8)
plt.legend(); plt.title("Commercial Liability: Tracking the Flood Hydrograph"); plt.show()

# posterior predictive trajectory
# in sample time series overlay (here hydrograph cuz water flow)

# only shows how well your model memorized the past (the data it trained on).
# needs az.compare or loo-cv to prove not hallucinating

In [ ]:


loo_arx = az.loo(trace_arx)
az.plot_khat(loo_arx, show_bins=True)
plt.title("Forensic Audit: Pareto k Diagnostic for Hydrological Anomalies")

In [ ]:
#basically we made a 2nd reality which did not have memory and verified that model which can identify memory via phi correctly identified as phi = 0